# Extensões (além do estudo pedido no enunciado)

Usa as mesmas funções de `src/` e grava nas mesmas pastas `results/` e `figures/` que o `pipeline.ipynb`,
então as tabelas e figuras finais incluem as extensões. Nada aqui refaz os treinos do estudo principal.

1. **Linha P — pixels crus:** os probes direto nos pixels (32×32×3 = 3072-d), sem encoder. Mostra quanto do
   desempenho de A vem da arquitetura e quanto já está nos pixels.
2. **Alunos extras (stretch goal):** tamanhos M (×1,0, 2,8M) e XL (×1,8, 9,1M) da mesma família, só no experimento
   principal (100% dos dados, 2 seeds, sem ablação), para a curva C − B × parâmetros ter 4 pontos.

**Pré-requisito:** a grade da SVM RBF já corrigida e os probes refeitos no `pipeline.ipynb`
(apague `results/probes/`, `results/clusters/` e `results/probe_hparams.json`, depois rode a célula da seção 9).

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from IPython.display import Image, display
from src import config as C
from src.common import DEVICE
from src.models import all_sizes, build_student, n_params
from src.extract import extract, all_encoders
from src.probe import select, run, HP_PATH
from src.train import train
from src import cost
from src.report import run as report, load_probes, summarize

assert hasattr(C, "EXTRA_SIZES"), "adicione EXTRA_SIZES / EXTRA_SEEDS ao src/config.py (ver README)"
assert HP_PATH.exists(), "rode antes a seção 9 do pipeline.ipynb (seleção + probes com a grade nova)"
print(json.loads(HP_PATH.read_text()))
print("tamanhos:", all_sizes(), "| seeds extras:", C.EXTRA_SEEDS)

## 1. Linha P — pixels crus
Mesma transformação determinística das outras linhas (imagem de 128 px → resize → normalização), só que
para 32×32 e sem rede: o vetor de pixels vai direto para L2 + StandardScaler + probes, com os mesmos
subconjuntos rotulados e os mesmos C/γ. Sem treino; alguns minutos de CPU.

In [ ]:
extract("P")
display(pd.Series(json.loads((C.RESULTS / "sanity_P.json").read_text())))
run("P")

In [ ]:
# probe linear: P contra as linhas do estudo principal (100% dos dados não rotulados)
s = summarize(load_probes())
t = s[(s.probe == "linear") & (s.data_frac == 1.0) & s["size"].isin(["-", "S", "L"])].copy()
t["v"] = (100 * t["mean"]).round(1).astype(str) + " ± " + (100 * t["std"].fillna(0)).round(1).astype(str)
tab = t.pivot_table(index=["row", "size"], columns="label_frac", values="v", aggfunc="first")
tab.columns = [f"{int(c * 100)}%" for c in tab.columns]
display(tab)
cl = pd.DataFrame([json.loads(p.read_text()) for p in (C.RESULTS / "clusters").glob("*.json")])
display(cl[cl.row.isin(["P", "A", "D"])][["tag", "feat_dim", "kmeans_nmi", "kmeans_ari"]]
        .sort_values("tag").round(3))

## 2. Alunos extras: M e XL
Mesma família (só a largura muda), mesmo orçamento e protocolo do estudo principal (40 épocas, mesmas
views em cache, mesmos probes), 2 seeds cada. Estimativa na RTX 3050: ~4–5 h no total — rode de madrugada.
Tudo é cacheado e retomável: se cair, rode a célula de novo.

In [ ]:
rows = []
for z, m in sorted(all_sizes().items(), key=lambda kv: kv[1]):
    net = build_student(z).to(DEVICE)
    rows.append({"size": z, "width_mult": m, "widths": net.widths,
                 "params_M": round(n_params(net) / 1e6, 2),
                 "gflops": round(cost.flops_per_image(net, C.IMG_SIZE) / 1e9, 3),
                 "feat_dim": net.num_features,
                 "extensão": z in C.EXTRA_SIZES})
pd.DataFrame(rows)

In [ ]:
for size, seeds in C.EXTRA_SEEDS.items():
    for s in seeds:
        for row in ("B", "C"):
            train(row, size, s)
            torch.cuda.empty_cache()

In [ ]:
# features (A das seeds extras + os novos B/C), probes, custo
for j in all_encoders():
    extract(*j)
for p in sorted(C.FEATS.glob("*.npz")):
    run(p.stem)
cost.main()

## 3. Relatório atualizado (inclui P e os 4 tamanhos)

In [ ]:
report("umap", "S")
g = pd.read_csv(C.RESULTS / "gap.csv")
g = g[(g.probe == "linear") & (g.data_frac == 1.0)].copy()
order = sorted(all_sizes(), key=all_sizes().get)
g["size"] = pd.Categorical(g["size"], order, ordered=True)
g["v"] = (100 * g["mean"]).round(2).astype(str) + " ± " + (100 * g["std"].fillna(0)).round(2).astype(str)
gt = g.pivot_table(index="size", columns="label_frac", values="v", aggfunc="first", observed=True)
gt.columns = [f"{int(c * 100)}%" for c in gt.columns]
print("C − B (pp), probe linear, 100% dos dados não rotulados")
display(gt)
for f in ["label_efficiency", "gap_vs_params"]:
    display(Image(str(C.FIGS / f"{f}.png")))